In [1]:
# =========================================
# CELL 1 — IMPORTS
# Vision + Audio Ablation Notebook
# =========================================

import os
import re
import torch
import numpy as np
import pandas as pd

from tqdm import tqdm

import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

from transformers import CLIPModel
from transformers import AutoTokenizer, AutoModel

from sklearn.metrics import accuracy_score, f1_score, classification_report

from torch.amp import autocast, GradScaler

from PIL import Image
from pathlib import Path

import matplotlib.pyplot as plt

from torchvision import transforms

In [2]:
# =========================================
# CELL 2 — DEVICE SETUP
# =========================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU Memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

Device: cuda
GPU: Tesla P100-PCIE-16GB
GPU Memory (GB): 17.06


In [3]:
# =========================================
# CELL 3 — DATASET PATHS + METADATA
# =========================================

DATASET_PATH = "/kaggle/input/datasets/ajfaisal002/crossmodal-misleading-video-dataset"

TRAIN_FRAMES = os.path.join(DATASET_PATH, "dataset_kaggle/extracted_frames/train")
TEST_FRAMES  = os.path.join(DATASET_PATH, "dataset_kaggle/extracted_frames/test")

TRAIN_AUDIO = os.path.join(DATASET_PATH, "extracted_audio/extracted_audio/train")
TEST_AUDIO  = os.path.join(DATASET_PATH, "extracted_audio/extracted_audio/test")

TRAIN_META = os.path.join(DATASET_PATH, "dataset_kaggle/extracted_text/train_metadata.csv")
TEST_META  = os.path.join(DATASET_PATH, "dataset_kaggle/extracted_text/test_metadata.csv")

train_df = pd.read_csv(TRAIN_META)
test_df  = pd.read_csv(TEST_META)

print("Train samples:", len(train_df))
print("Test samples:", len(test_df))

print("\nColumns:")
print(train_df.columns)

print("\nExample row:")
display(train_df.sample(5))

Train samples: 1600
Test samples: 800

Columns:
Index(['video_id', 'has_audio', 'transcript', 'ocr_text', 'category',
       'subcategory'],
      dtype='object')

Example row:


,video_id,has_audio,transcript,ocr_text,category,subcategory
464,SUS_065,True,Gravity doesn't affect me anymore.,NaN,misleading,scientifically_unrealistic_scene
1406,SAFE_607,True,Wegenbau.,"F F = F = G F - G( m F = G ( m, m F = G(m1 m2)...",safe,NaN
346,PM_147,True,It's fine.,NaN,misleading,perception_manipulation
940,SAFE_141,True,Please take this.,PLEASE TAKE THIS,safe,NaN
97,IF_098,True,Prayer alone cures every disease.,PRAYER ALONE CURES EVERY DISEASE,misleading,identity_fabrication


In [4]:
# =========================================
# CELL 4 — LABELS + INPUT MODE
# =========================================

label_names = [
    "safe",
    "identity_fabrication",
    "perception_manipulation",
    "scientifically_unrealistic_scene",
    "surreal_content"
]

label2id = {k:i for i,k in enumerate(label_names)}
id2label = {i:k for k,i in label2id.items()}

print("Label mapping ready.")


# -----------------------------------------
# INPUT MODE
# -----------------------------------------
# "vision_asr"   → Vision + ASR transcript
# "vision_audio" → Vision + Audio embeddings

INPUT_MODE = "vision_asr"

print("\nInput Mode:", INPUT_MODE)

Label mapping ready.

Input Mode: vision_asr


In [5]:
# =========================================
# CELL 5 — LOAD ENCODERS
# =========================================

print("Loading encoders...")

# ---------- CLIP (Vision) ----------
clip = CLIPModel.from_pretrained(
    "openai/clip-vit-base-patch32"
).to(device)

# ---------- RoBERTa (for ASR mode) ----------
tokenizer = AutoTokenizer.from_pretrained(
    "roberta-base"
)

roberta = AutoModel.from_pretrained(
    "roberta-base"
).to(device)

# Freeze encoders to save GPU memory
for p in clip.parameters():
    p.requires_grad = False

for p in roberta.parameters():
    p.requires_grad = False

clip.eval()
roberta.eval()

print("Encoders loaded and frozen.")

print("\nCLIP hidden size:", clip.config.vision_config.hidden_size)
print("RoBERTa hidden size:", roberta.config.hidden_size)

Loading encoders...


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Encoders loaded and frozen.

CLIP hidden size: 768
RoBERTa hidden size: 768


In [6]:
# =========================================
# CELL 6 — SAFE EMBEDDING FUNCTIONS
# =========================================

# ---------- CLIP Vision Embedding ----------
def clip_vision_embed(pixel_values):
    """
    Stable CLIP vision embedding
    Works across HuggingFace versions
    """

    outputs = clip.vision_model(pixel_values=pixel_values)

    if hasattr(outputs, "pooler_output") and outputs.pooler_output is not None:
        return outputs.pooler_output

    return outputs.last_hidden_state.mean(dim=1)


# ---------- RoBERTa Text Embedding ----------
def roberta_text_embed(texts):
    """
    Extract CLS embedding from ASR transcripts
    """

    tokens = tokenizer(
        list(texts),
        padding=True,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    outputs = roberta(**tokens)

    return outputs.last_hidden_state[:, 0, :]


print("Embedding functions ready.")

Embedding functions ready.


In [7]:
# =========================================
# CELL 7 — FRAME LOADER
# =========================================

frame_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])


def load_frames(video_id, frame_dir):
    """
    Load 16 frames for a video
    Output shape: (16,3,224,224)
    """

    frames = []

    video_folder = os.path.join(frame_dir, video_id)

    for i in range(1,17):

        frame_path = os.path.join(
            video_folder,
            f"frame_{i:02d}.jpg"
        )

        if os.path.exists(frame_path):

            img = Image.open(frame_path).convert("RGB")
            img = frame_transform(img)

        else:
            img = torch.zeros(3,224,224)

        frames.append(img)

    frames = torch.stack(frames)

    return frames


print("Frame loader ready.")

Frame loader ready.


In [8]:
# =========================================
# CELL 8 — AUDIO PROCESSING (MFCC)
# =========================================

import librosa


def extract_mfcc(audio_path):
    """
    Convert audio waveform → MFCC
    Output shape: (40,160)
    """

    try:
        y, sr = librosa.load(audio_path, sr=16000)

        mfcc = librosa.feature.mfcc(
            y=y,
            sr=sr,
            n_mfcc=40
        )

        # pad / truncate to 160 time steps
        if mfcc.shape[1] < 160:

            pad_width = 160 - mfcc.shape[1]

            mfcc = np.pad(
                mfcc,
                ((0,0),(0,pad_width)),
                mode="constant"
            )

        else:
            mfcc = mfcc[:, :160]

    except:
        mfcc = np.zeros((40,160))

    return mfcc


def load_audio(video_id, audio_dir):

    audio_path = os.path.join(audio_dir, f"{video_id}.wav")

    mfcc = extract_mfcc(audio_path)

    return torch.tensor(mfcc, dtype=torch.float32)


print("Audio processing ready.")

Audio processing ready.


In [9]:
# =========================================
# CELL 9 — UNIFIED VIDEO DATASET
# =========================================

class VideoDataset(Dataset):

    def __init__(self, df, frame_dir, audio_dir=None):
        self.df = df.reset_index(drop=True)
        self.frame_dir = frame_dir
        self.audio_dir = audio_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        video_id = row["video_id"]

        # ---------- Frames ----------
        frames = load_frames(video_id, self.frame_dir)

        # ---------- ASR Text ----------
        transcript = str(row.get("transcript",""))

        # ---------- Audio ----------
        audio = None
        if self.audio_dir is not None:
            audio = load_audio(video_id, self.audio_dir)

        label = int(row["label"])

        return {
            "frames": frames,        # (16,3,224,224)
            "text": transcript,      # ASR text
            "audio": audio,          # MFCC (40,160)
            "label": label
        }


print("Unified dataset ready.")

Unified dataset ready.


In [10]:
# =========================================
# CELL 10 — HIERARCHICAL DATASET SPLIT
# =========================================

print("Preparing hierarchical datasets...")

# ---------- Stage-1 ----------
stage1_df = train_df.copy()

def stage1_label(row):
    if row["category"] == "safe":
        return 0
    return 1

stage1_df["label"] = stage1_df.apply(stage1_label, axis=1)

print("\nStage-1 distribution:")
print(stage1_df["label"].value_counts())


# ---------- Stage-2 ----------
stage2_df = train_df[train_df["category"] != "safe"].copy()

stage2_label_map = {
    "identity_fabrication": 0,
    "perception_manipulation": 1,
    "scientifically_unrealistic_scene": 2,
    "surreal_content": 3
}

stage2_df["label"] = stage2_df["subcategory"].map(stage2_label_map)

print("\nStage-2 distribution:")
print(stage2_df["label"].value_counts())


# ---------- Dataset objects ----------
stage1_dataset = VideoDataset(
    stage1_df,
    TRAIN_FRAMES,
    TRAIN_AUDIO
)

stage2_dataset = VideoDataset(
    stage2_df,
    TRAIN_FRAMES,
    TRAIN_AUDIO
)


# ---------- DataLoaders ----------
stage1_loader = DataLoader(
    stage1_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=2
)

stage2_loader = DataLoader(
    stage2_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=2
)

print("\nDataLoaders ready.")

Preparing hierarchical datasets...

Stage-1 distribution:
label
1    800
0    800
Name: count, dtype: int64

Stage-2 distribution:
label
0    200
1    200
2    200
3    200
Name: count, dtype: int64

DataLoaders ready.


In [11]:
# =========================================
# CELL 11 — FLEXIBLE MULTIMODAL MODEL
# =========================================

class VisionASRModel(nn.Module):

    def __init__(self, num_classes):

        super().__init__()

        img_dim = 768
        txt_dim = 768

        fusion_dim = img_dim + txt_dim

        self.img_gate = nn.Linear(img_dim, img_dim)
        self.txt_gate = nn.Linear(txt_dim, txt_dim)

        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, frames, texts):

        B = frames.size(0)

        # Vision
        frames = frames.view(-1,3,224,224).to(device)

        with torch.no_grad():
            img_feat = clip_vision_embed(frames)

        img_feat = img_feat.view(B,16,-1).mean(dim=1)

        # Text
        with torch.no_grad():
            txt_feat = roberta_text_embed(texts)

        # Normalization
        img_feat = F.layer_norm(img_feat, img_feat.shape[1:])
        txt_feat = F.layer_norm(txt_feat, txt_feat.shape[1:])

        # Gated fusion
        g_img = torch.sigmoid(self.img_gate(img_feat))
        g_txt = torch.sigmoid(self.txt_gate(txt_feat))

        img_feat = img_feat * g_img
        txt_feat = txt_feat * g_txt

        fusion = torch.cat([img_feat, txt_feat], dim=1)

        return self.classifier(fusion)



class VisionAudioModel(nn.Module):

    def __init__(self, num_classes):

        super().__init__()

        img_dim = 768
        audio_dim = 128

        # Audio CNN
        self.audio_encoder = nn.Sequential(

            nn.Conv2d(1,16,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16,32,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Flatten(),
            nn.Linear(32*10*40,128),
            nn.ReLU()
        )

        fusion_dim = img_dim + audio_dim

        self.img_gate = nn.Linear(img_dim,img_dim)
        self.audio_gate = nn.Linear(audio_dim,audio_dim)

        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim,512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512,num_classes)
        )

    def forward(self, frames, audio):

        B = frames.size(0)

        # Vision
        frames = frames.view(-1,3,224,224).to(device)

        with torch.no_grad():
            img_feat = clip_vision_embed(frames)

        img_feat = img_feat.view(B,16,-1).mean(dim=1)

        img_feat = F.layer_norm(img_feat, img_feat.shape[1:])

        # Audio
        audio = audio.unsqueeze(1).to(device)

        audio_feat = self.audio_encoder(audio)

        audio_feat = F.layer_norm(audio_feat, audio_feat.shape[1:])

        # Gated fusion
        g_img = torch.sigmoid(self.img_gate(img_feat))
        g_audio = torch.sigmoid(self.audio_gate(audio_feat))

        img_feat = img_feat * g_img
        audio_feat = audio_feat * g_audio

        fusion = torch.cat([img_feat,audio_feat],dim=1)

        return self.classifier(fusion)


print("Multimodal models ready.")

Multimodal models ready.


In [12]:
# =========================================
# CELL 12 — INITIALIZE HIERARCHICAL MODELS
# =========================================

print("Initializing models for:", INPUT_MODE)


# ---------- Stage-1 ----------
if INPUT_MODE == "vision_asr":

    binary_model = VisionASRModel(num_classes=2).to(device)

else:

    binary_model = VisionAudioModel(num_classes=2).to(device)


# ---------- Stage-2 ----------
if INPUT_MODE == "vision_asr":

    stage2_model = VisionASRModel(num_classes=4).to(device)

else:

    stage2_model = VisionAudioModel(num_classes=4).to(device)


print("Stage-1 model created.")
print("Stage-2 model created.")


# ---------- Loss + Optimizers ----------
criterion_stage1 = nn.CrossEntropyLoss()
criterion_stage2 = nn.CrossEntropyLoss()

optimizer_stage1 = optim.Adam(binary_model.parameters(), lr=1e-3)
optimizer_stage2 = optim.Adam(stage2_model.parameters(), lr=1e-3)

scaler = GradScaler()

print("Optimizers ready.")

Initializing models for: vision_asr
Stage-1 model created.
Stage-2 model created.
Optimizers ready.


In [13]:
# =========================================
# CELL 13 — TRAIN STAGE-1
# SAFE vs MISLEADING
# =========================================

EPOCHS = 5

print("\nTraining Stage-1\n")

for epoch in range(EPOCHS):

    binary_model.train()

    total_loss = 0
    all_preds = []
    all_labels = []

    for batch in tqdm(stage1_loader):

        frames = batch["frames"]
        labels = batch["label"].to(device)

        optimizer_stage1.zero_grad()

        with autocast(device_type="cuda"):

            if INPUT_MODE == "vision_asr":

                texts = batch["text"]

                outputs = binary_model(frames, texts)

            else:

                audio = batch["audio"]

                outputs = binary_model(frames, audio)

            loss = criterion_stage1(outputs, labels)

        scaler.scale(loss).backward()

        scaler.step(optimizer_stage1)

        scaler.update()

        total_loss += loss.item()

        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)

    f1 = f1_score(all_labels, all_preds)

    print(
        f"\nEpoch {epoch+1}/{EPOCHS} | "
        f"Loss: {total_loss/len(stage1_loader):.4f} | "
        f"Acc: {acc:.4f} | "
        f"F1: {f1:.4f}"
    )


Training Stage-1



100%|██████████| 400/400 [03:12<00:00,  2.08it/s]



Epoch 1/5 | Loss: 0.2530 | Acc: 0.8962 | F1: 0.8968


100%|██████████| 400/400 [02:14<00:00,  2.98it/s]



Epoch 2/5 | Loss: 0.0925 | Acc: 0.9669 | F1: 0.9669


...


Epoch 3/5 | Loss: 0.0446 | Acc: 0.9831 | F1: 0.9832


100%|██████████| 400/400 [02:14<00:00,  2.98it/s]



Epoch 4/5 | Loss: 0.0192 | Acc: 0.9938 | F1: 0.9938


100%|██████████| 400/400 [02:14<00:00,  2.97it/s]


Epoch 5/5 | Loss: 0.0122 | Acc: 0.9956 | F1: 0.9956


In [14]:
# =========================================
# CELL 14 — TRAIN STAGE-2
# MISINFORMATION TYPE CLASSIFIER
# =========================================

EPOCHS = 5

print("\nTraining Stage-2\n")

for epoch in range(EPOCHS):

    stage2_model.train()

    total_loss = 0
    all_preds = []
    all_labels = []

    for batch in tqdm(stage2_loader):

        frames = batch["frames"]
        labels = batch["label"].to(device)

        optimizer_stage2.zero_grad()

        with autocast(device_type="cuda"):

            if INPUT_MODE == "vision_asr":

                texts = batch["text"]
                outputs = stage2_model(frames, texts)

            else:

                audio = batch["audio"]
                outputs = stage2_model(frames, audio)

            loss = criterion_stage2(outputs, labels)

        scaler.scale(loss).backward()

        scaler.step(optimizer_stage2)

        scaler.update()

        total_loss += loss.item()

        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)

    f1 = f1_score(all_labels, all_preds, average="macro")

    print(
        f"\nEpoch {epoch+1}/{EPOCHS} | "
        f"Loss: {total_loss/len(stage2_loader):.4f} | "
        f"Acc: {acc:.4f} | "
        f"MacroF1: {f1:.4f}"
    )


Training Stage-2



100%|██████████| 200/200 [01:07<00:00,  2.98it/s]



Epoch 1/5 | Loss: 0.4753 | Acc: 0.8113 | MacroF1: 0.8093


100%|██████████| 200/200 [01:04<00:00,  3.08it/s]



Epoch 2/5 | Loss: 0.1121 | Acc: 0.9637 | MacroF1: 0.9638


...


Epoch 3/5 | Loss: 0.0492 | Acc: 0.9862 | MacroF1: 0.9863


100%|██████████| 200/200 [01:05<00:00,  3.07it/s]



Epoch 4/5 | Loss: 0.0195 | Acc: 0.9962 | MacroF1: 0.9963


100%|██████████| 200/200 [01:05<00:00,  3.06it/s]


Epoch 5/5 | Loss: 0.0043 | Acc: 0.9975 | MacroF1: 0.9975


In [15]:
# =========================================
# CELL 15 — HIERARCHICAL PREDICTION
# =========================================

import numpy as np

@torch.no_grad()
def hierarchical_predict(frames16, text=None, audio=None):

    binary_model.eval()
    stage2_model.eval()

    frames = frames16.unsqueeze(0).to(device)

    # ---------- Stage 1 ----------
    if INPUT_MODE == "vision_asr":

        logits_stage1 = binary_model(frames, [text])

    else:

        audio = audio.unsqueeze(0).to(device)
        logits_stage1 = binary_model(frames, audio)

    probs_stage1 = torch.softmax(logits_stage1, dim=1)[0].cpu().numpy()

    pred_stage1 = int(np.argmax(probs_stage1))

    # SAFE
    if pred_stage1 == 0:

        return {
            "label": "safe",
            "confidence": float(probs_stage1[0]),
            "stage1_probs": probs_stage1
        }

    # ---------- Stage 2 ----------
    if INPUT_MODE == "vision_asr":

        logits_stage2 = stage2_model(frames, [text])

    else:

        logits_stage2 = stage2_model(frames, audio)

    probs_stage2 = torch.softmax(logits_stage2, dim=1)[0].cpu().numpy()

    pred_stage2 = int(np.argmax(probs_stage2))

    mis_map = {
        0: "identity_fabrication",
        1: "perception_manipulation",
        2: "scientifically_unrealistic_scene",
        3: "surreal_content"
    }

    return {
        "label": mis_map[pred_stage2],
        "confidence": float(probs_stage2[pred_stage2]),
        "stage1_probs": probs_stage1,
        "stage2_probs": probs_stage2
    }


print("Hierarchical prediction ready.")

Hierarchical prediction ready.


In [16]:
# =========================================
# CELL 16 — TEST EVALUATION
# =========================================

print("\nEvaluating hierarchical model on TEST set...\n")

all_preds = []
all_labels = []

for i in tqdm(range(len(test_df))):

    row = test_df.iloc[i]

    video_id = row["video_id"]

    transcript = str(row.get("transcript",""))

    frames = load_frames(video_id, TEST_FRAMES)

    if INPUT_MODE == "vision_asr":

        result = hierarchical_predict(frames, transcript)

    else:

        audio = load_audio(video_id, TEST_AUDIO)

        result = hierarchical_predict(frames, audio=audio)

    pred_label = result["label"]

    # true label
    if row["category"] == "safe":
        true_label = "safe"
    else:
        true_label = row["subcategory"]

    all_preds.append(pred_label)
    all_labels.append(true_label)


pred_ids = [label2id[p] for p in all_preds]
true_ids = [label2id[t] for t in all_labels]

acc = accuracy_score(true_ids, pred_ids)

f1 = f1_score(true_ids, pred_ids, average="macro")

print("\n==============================")
print("TEST RESULTS")
print("==============================")

print("Accuracy:", acc)
print("Macro F1:", f1)

print("\nClassification Report:\n")

print(
    classification_report(
        true_ids,
        pred_ids,
        target_names=label_names
    )
)


Evaluating hierarchical model on TEST set...



100%|██████████| 800/800 [02:11<00:00,  6.06it/s]


TEST RESULTS
Accuracy: 0.725
Macro F1: 0.678699776414583

Classification Report:

                                  precision    recall  f1-score   support

                            safe       0.78      0.81      0.79       400
            identity_fabrication       0.62      0.43      0.51       100
         perception_manipulation       0.75      0.64      0.69       100
scientifically_unrealistic_scene       0.55      0.89      0.68       100
                 surreal_content       0.90      0.60      0.72       100

                        accuracy                           0.72       800
                       macro avg       0.72      0.67      0.68       800
                    weighted avg       0.74      0.72      0.72       800



In [17]:
# =========================================
# CELL 17 — SWITCH TO AUDIO EXPERIMENT
# =========================================

INPUT_MODE = "vision_audio"

print("Switched experiment mode to:", INPUT_MODE)

Switched experiment mode to: vision_audio


In [18]:
# =========================================
# CELL 18 — INITIALIZE AUDIO MODELS
# =========================================

print("Initializing Vision + Audio hierarchical models...")

binary_model_audio = VisionAudioModel(num_classes=2).to(device)
stage2_model_audio = VisionAudioModel(num_classes=4).to(device)

criterion_stage1_audio = nn.CrossEntropyLoss()
criterion_stage2_audio = nn.CrossEntropyLoss()

optimizer_stage1_audio = optim.Adam(binary_model_audio.parameters(), lr=1e-3)
optimizer_stage2_audio = optim.Adam(stage2_model_audio.parameters(), lr=1e-3)

scaler_audio = GradScaler()

print("Audio hierarchical models ready.")

Initializing Vision + Audio hierarchical models...
Audio hierarchical models ready.


In [19]:
# =========================================
# CELL 19 — TRAIN STAGE-1 (VISION + AUDIO)
# =========================================

EPOCHS = 5

print("\nTraining Stage-1 (Vision + Audio)\n")

for epoch in range(EPOCHS):

    binary_model_audio.train()

    total_loss = 0
    all_preds = []
    all_labels = []

    for batch in tqdm(stage1_loader):

        frames = batch["frames"]
        audio = batch["audio"]
        labels = batch["label"].to(device)

        optimizer_stage1_audio.zero_grad()

        with autocast(device_type="cuda"):

            outputs = binary_model_audio(frames, audio)

            loss = criterion_stage1_audio(outputs, labels)

        scaler_audio.scale(loss).backward()

        scaler_audio.step(optimizer_stage1_audio)
        scaler_audio.update()

        total_loss += loss.item()

        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)

    print(
        f"\nEpoch {epoch+1}/{EPOCHS} | "
        f"Loss: {total_loss/len(stage1_loader):.4f} | "
        f"Acc: {acc:.4f} | "
        f"F1: {f1:.4f}"
    )


Training Stage-1 (Vision + Audio)



100%|██████████| 400/400 [02:12<00:00,  3.03it/s]



Epoch 1/5 | Loss: 0.2572 | Acc: 0.8869 | F1: 0.8881


100%|██████████| 400/400 [02:14<00:00,  2.98it/s]



Epoch 2/5 | Loss: 0.0910 | Acc: 0.9663 | F1: 0.9663


...


Epoch 3/5 | Loss: 0.0351 | Acc: 0.9869 | F1: 0.9869


100%|██████████| 400/400 [02:15<00:00,  2.95it/s]



Epoch 4/5 | Loss: 0.0211 | Acc: 0.9919 | F1: 0.9919


100%|██████████| 400/400 [02:10<00:00,  3.06it/s]



Epoch 5/5 | Loss: 0.0106 | Acc: 0.9981 | F1: 0.9981


In [20]:
# =========================================
# CELL 20 — TRAIN STAGE-2 (VISION + AUDIO)
# =========================================

print("\nTraining Stage-2 (Vision + Audio)\n")

for epoch in range(EPOCHS):

    stage2_model_audio.train()

    total_loss = 0
    all_preds = []
    all_labels = []

    for batch in tqdm(stage2_loader):

        frames = batch["frames"]
        audio = batch["audio"]
        labels = batch["label"].to(device)

        optimizer_stage2_audio.zero_grad()

        with autocast(device_type="cuda"):

            outputs = stage2_model_audio(frames, audio)

            loss = criterion_stage2_audio(outputs, labels)

        scaler_audio.scale(loss).backward()

        scaler_audio.step(optimizer_stage2_audio)
        scaler_audio.update()

        total_loss += loss.item()

        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average="macro")

    print(
        f"\nEpoch {epoch+1}/{EPOCHS} | "
        f"Loss: {total_loss/len(stage2_loader):.4f} | "
        f"Acc: {acc:.4f} | "
        f"MacroF1: {f1:.4f}"
    )


Training Stage-2 (Vision + Audio)



100%|██████████| 200/200 [01:05<00:00,  3.06it/s]



Epoch 1/5 | Loss: 0.4680 | Acc: 0.8237 | MacroF1: 0.8223


100%|██████████| 200/200 [01:04<00:00,  3.08it/s]



Epoch 2/5 | Loss: 0.1335 | Acc: 0.9600 | MacroF1: 0.9599


...


Epoch 3/5 | Loss: 0.0536 | Acc: 0.9800 | MacroF1: 0.9800


100%|██████████| 200/200 [01:03<00:00,  3.13it/s]



Epoch 4/5 | Loss: 0.0166 | Acc: 0.9950 | MacroF1: 0.9950


100%|██████████| 200/200 [01:04<00:00,  3.10it/s]


Epoch 5/5 | Loss: 0.0025 | Acc: 0.9988 | MacroF1: 0.9987


In [21]:
# =========================================
# CELL 21 — HIERARCHICAL PREDICTION AUDIO
# =========================================

@torch.no_grad()
def hierarchical_predict_audio(frames16, audio):

    binary_model_audio.eval()
    stage2_model_audio.eval()

    frames = frames16.unsqueeze(0).to(device)
    audio = audio.unsqueeze(0).to(device)

    logits_stage1 = binary_model_audio(frames, audio)
    probs_stage1 = torch.softmax(logits_stage1, dim=1)[0].cpu().numpy()

    pred_stage1 = int(np.argmax(probs_stage1))

    if pred_stage1 == 0:

        return {
            "label": "safe",
            "confidence": float(probs_stage1[0]),
            "stage1_probs": probs_stage1
        }

    logits_stage2 = stage2_model_audio(frames, audio)
    probs_stage2 = torch.softmax(logits_stage2, dim=1)[0].cpu().numpy()

    pred_stage2 = int(np.argmax(probs_stage2))

    mis_map = {
        0: "identity_fabrication",
        1: "perception_manipulation",
        2: "scientifically_unrealistic_scene",
        3: "surreal_content"
    }

    return {
        "label": mis_map[pred_stage2],
        "confidence": float(probs_stage2[pred_stage2]),
        "stage1_probs": probs_stage1,
        "stage2_probs": probs_stage2
    }

print("Audio hierarchical predictor ready.")

Audio hierarchical predictor ready.


In [22]:
# =========================================
# CELL 22 — TEST EVALUATION (VISION + AUDIO)
# =========================================

print("\nEvaluating Vision + Audio model on TEST set...\n")

all_preds = []
all_labels = []

for i in tqdm(range(len(test_df))):

    row = test_df.iloc[i]

    video_id = row["video_id"]

    frames = load_frames(video_id, TEST_FRAMES)
    audio = load_audio(video_id, TEST_AUDIO)

    result = hierarchical_predict_audio(frames, audio)

    pred_label = result["label"]

    if row["category"] == "safe":
        true_label = "safe"
    else:
        true_label = row["subcategory"]

    all_preds.append(pred_label)
    all_labels.append(true_label)

pred_ids = [label2id[p] for p in all_preds]
true_ids = [label2id[t] for t in all_labels]

acc = accuracy_score(true_ids, pred_ids)
f1 = f1_score(true_ids, pred_ids, average="macro")

print("\n==============================")
print("VISION + AUDIO RESULTS")
print("==============================")

print("Accuracy:", acc)
print("Macro F1:", f1)

print("\nClassification Report:\n")

print(
    classification_report(
        true_ids,
        pred_ids,
        target_names=label_names
    )
)


Evaluating Vision + Audio model on TEST set...



100%|██████████| 800/800 [01:57<00:00,  6.80it/s]


VISION + AUDIO RESULTS
Accuracy: 0.73
Macro F1: 0.679501957653228

Classification Report:

                                  precision    recall  f1-score   support

                            safe       0.76      0.84      0.80       400
            identity_fabrication       0.74      0.40      0.52       100
         perception_manipulation       0.66      0.71      0.68       100
scientifically_unrealistic_scene       0.57      0.74      0.65       100
                 surreal_content       0.93      0.63      0.75       100

                        accuracy                           0.73       800
                       macro avg       0.73      0.66      0.68       800
                    weighted avg       0.74      0.73      0.72       800

